# Entrenar YOLO para Riachuelo (chanchito blanco + melaza/fumagina)

QUÉ HACE: entrena el modelo de la plataforma de monitoreo en Google Colab y deja en Google Drive el `modelo.onnx` y la configuración para `/gestion/` → Modelos de IA.

Antes de empezar:
1. **Entorno de ejecución → Cambiar tipo de entorno → GPU T4.**
2. En tu Google Drive crea la carpeta `riachuelo_yolo/` con:
   - `externas/` → los .zip de Roboflow Universe (formato YOLO26 o YOLOv8).
   - `propias/` → los .zip exportados de tu proyecto de etiquetado (formato YOLO), con las clases escritas exactamente `chanchito_blanco` y `melaza_fumagina`.
3. Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Si Colab se desconecta, vuelve a ejecutar todo: el entrenamiento continúa desde el último punto guardado.

Guía completa: doc «Guía YOLO — Chanchito blanco (Riachuelo)».

In [ ]:
# 1. Comprobar la GPU (debe decir Tesla T4 o similar)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Configuración (lo único que se edita)
BASE = '/content/drive/MyDrive/riachuelo_yolo'   # carpeta en Drive
MODELO_BASE = 'yolo26n.pt'      # alternativa: 'yolo11n.pt'. Para reentrenar: ruta a un best.pt anterior en Drive
VERSION = '2026.10-v1'          # única por entrenamiento; es la misma que irá en /gestion/
CLASES = ['chanchito_blanco', 'melaza_fumagina']   # ORDEN FIJO: igual en /gestion/
TILE, OVERLAP = 1280, 0.2       # recortes: iguales al campo tiling del worker
EPOCAS, PACIENCIA, LOTE = 150, 30, 16
MAPA = ['mealybug=chanchito_blanco', 'mealybugs=chanchito_blanco']  # clases de internet -> nuestras
RAMA = 'prueba2'

In [ ]:
# 4. Instalar dependencias y traer el código de la plataforma (scripts tools/ia y el detector del worker)
!pip -q install "ultralytics>=8.4" onnx onnxslim onnxruntime "django>=5.2,<6"
!rm -rf /content/plat && git clone -q --depth 1 -b {RAMA} https://github.com/OskitarBB/agricola-riachuelo-platform.git /content/plat
!ls /content/plat/tools/ia

In [ ]:
# 5. Descomprimir los datasets de Drive en el disco rápido de Colab
import glob, os, zipfile, shutil
shutil.rmtree('/content/datos', ignore_errors=True)
fuentes = {'externas': [], 'propias': []}
for tipo in fuentes:
    for z in sorted(glob.glob(f'{BASE}/{tipo}/*.zip')):
        destino = f"/content/datos/{tipo}/{os.path.splitext(os.path.basename(z))[0].replace(' ', '_')}"
        zipfile.ZipFile(z).extractall(destino)
        fuentes[tipo].append(destino)
        print(tipo, '->', destino)
assert fuentes['externas'] or fuentes['propias'], f'No hay .zip en {BASE}/externas ni {BASE}/propias'
if not fuentes['propias']:
    print('AVISO: sin fotos propias, val se arma con el 15 % de las externas y no hay prueba real. Sirve solo para probar la integración.')

In [ ]:
# 6. Cortar en recortes de 1280 px y dividir train/val/test (mismo corte que el worker)
args = ['--salida', '/content/dataset', '--tile', str(TILE), '--overlap', str(OVERLAP), '--clases', *CLASES]
if fuentes['propias']:
    args += ['--propias', *fuentes['propias']]
if fuentes['externas']:
    args += ['--externas', *fuentes['externas']]
for m in MAPA:
    args += ['--mapa', m]
!cd /content/plat && python tools/ia/cortar_mosaicos.py {' '.join(args)}
os.makedirs(f'{BASE}/runs/{VERSION}', exist_ok=True)
shutil.copy('/content/dataset/resumen.json', f'{BASE}/runs/{VERSION}/resumen_dataset.json')

In [ ]:
# 7. Entrenar (continúa solo si Colab se desconectó a la mitad)
from ultralytics import YOLO
corrida = f'{BASE}/runs/{VERSION}'
ultimo = f'{corrida}/weights/last.pt'
if os.path.exists(ultimo):
    print('Continuando desde', ultimo)
    YOLO(ultimo).train(resume=True)
else:
    YOLO(MODELO_BASE).train(
        data='/content/dataset/data.yaml', imgsz=640, epochs=EPOCAS, patience=PACIENCIA, batch=LOTE,
        project=f'{BASE}/runs', name=VERSION, exist_ok=True,
        degrees=10, fliplr=0.5, flipud=0.0, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, mosaic=1.0, close_mosaic=10,
        seed=2026, plots=True)

In [ ]:
# 8. Medir en el conjunto de prueba (recortes) con Ultralytics
mejor = f'{corrida}/weights/best.pt'
modelo = YOLO(mejor)
if os.listdir('/content/dataset/images/test'):
    m = modelo.val(data='/content/dataset/data.yaml', split='test', imgsz=640, plots=True)
    print('mAP50 por clase:', dict(zip(CLASES, [round(x, 3) for x in m.box.maps])))
else:
    print('Sin conjunto de prueba (faltan fotos propias).')

In [ ]:
# 9. Exportar a ONNX (entrada fija 640, salida cruda: el worker hace el NMS)
onnx = modelo.export(format='onnx', imgsz=640)
os.makedirs(f'{BASE}/modelos/{VERSION}', exist_ok=True)
shutil.copy(onnx, f'{BASE}/modelos/{VERSION}/modelo.onnx')
print('Guardado en', f'{BASE}/modelos/{VERSION}/modelo.onnx')

In [ ]:
# 10. Verificar el ONNX con el MISMO código del worker sobre fotos completas de prueba y elegir conf_threshold
salida = f'{BASE}/modelos/{VERSION}/verificacion'
pc = '/content/dataset/prueba_completa'
etq = f'--etiquetas {pc}/labels --barrido' if os.listdir(f'{pc}/images') else ''
imgs = f'{pc}/images' if etq else '/content/dataset/images/val'
!cd /content/plat && python tools/ia/verificar_onnx.py --modelo {BASE}/modelos/{VERSION}/modelo.onnx --imagenes {imgs} {etq} --clases {' '.join(CLASES)} --tile {TILE} --overlap {OVERLAP} --version {VERSION} --salida {salida} --limite 60

In [ ]:
# 11. Ver algunas fotos con las cajas que sugiere el modelo
from IPython.display import Image as Im, display
for f in sorted(glob.glob(f'{salida}/*_cajas.jpg'))[:6]:
    display(Im(f, width=700))

## Siguiente paso: activar en el VPS

En Drive quedan `modelos/<VERSION>/modelo.onnx` y `modelos/<VERSION>/verificacion/model_config.json`.

1. Descarga `modelo.onnx` a la PC y súbelo: `scp modelo.onnx root@179.198.103.82:/srv/riachuelo/deploy/modelos/modelo.onnx`
2. En `https://monitoreo.agricolariachuelo.org/gestion/` → Modelos de IA → Agregar: copia los valores de `model_config.json` (nombre, versión, sha256, clases, umbrales, imgsz, tiling) y marca **active**.
3. En el VPS: `cd /srv/riachuelo && bash deploy/actualizar.sh` y revisa `docker compose logs -f worker`.

Solo activar si la verificación detecta al menos el 85 % de las fotos de prueba con plaga.